En este ejemplo se protege el estado interno de `Medico`, `Paciente` y `Cita` mediante atributos privados (`__atributo`) y una interfaz pública controlada.

El usuario no modifica directamente la especialidad, el estado del paciente, la agenda ni el estado de una cita. Cada cambio pasa por una propiedad o un método que valida la operación.

## Diseño de las clases

- `Medico` encapsula `__especialidad` y `__consultorio`; sus propiedades validan que no estén vacíos.
- `Paciente` encapsula `__estado` y `__citas`; el estado solo acepta valores definidos y las citas se consultan como una tupla inmodificable.
- `Cita` encapsula `__estado`; solo puede cambiar mediante `cancelar()`.

Esta protección evita que código externo deje el objeto en un estado inconsistente.

In [1]:
from dataclasses import dataclass
from datetime import datetime


@dataclass
class Medico:
    nombre: str
    __especialidad: str
    __consultorio: str

    @property
    def especialidad(self) -> str:
        return self.__especialidad

    @especialidad.setter
    def especialidad(self, valor: str) -> None:
        if not valor.strip():
            raise ValueError("La especialidad no puede estar vacía")
        self.__especialidad = valor.strip()

    @property
    def consultorio(self) -> str:
        return self.__consultorio

    @consultorio.setter
    def consultorio(self, valor: str) -> None:
        if not valor.strip():
            raise ValueError("El consultorio no puede estar vacío")
        self.__consultorio = valor.strip()


@dataclass
class Cita:
    fecha: datetime
    medico: Medico
    __estado: str = "agendada"

    @property
    def estado(self) -> str:
        return self.__estado

    def cancelar(self) -> None:
        if self.__estado == "cancelada":
            raise ValueError("La cita ya está cancelada")
        self.__estado = "cancelada"


class Paciente:
    def __init__(self, nombre: str, eps: str) -> None:
        self.nombre = nombre
        self.eps = eps
        self.__estado = "activo"
        self.__citas: list[Cita] = []

    @property
    def estado(self) -> str:
        return self.__estado

    @estado.setter
    def estado(self, valor: str) -> None:
        estados_validos = {"activo", "inactivo"}
        if valor not in estados_validos:
            raise ValueError("El estado debe ser 'activo' o 'inactivo'")
        self.__estado = valor

    @property
    def citas(self) -> tuple[Cita, ...]:
        return tuple(self.__citas)

    def agendar_cita(self, medico: Medico, fecha: datetime) -> Cita:
        if self.__estado != "activo":
            raise ValueError("El paciente inactivo no puede agendar citas")
        cita = Cita(fecha, medico)
        self.__citas.append(cita)
        return cita

    def cancelar_cita(self, cita: Cita) -> None:
        if cita not in self.__citas:
            raise ValueError("La cita no pertenece al paciente")
        cita.cancelar()

In [ ]:
medico = Medico("Laura Gómez", "Medicina interna", "204")
paciente = Paciente("Carlos Pérez", "SaludTotal")
cita = paciente.agendar_cita(medico, datetime(2026, 9, 15, 10, 30))

print(f"Especialidad: {medico.especialidad}")
print(f"Estado inicial: {paciente.estado}")
print(f"Citas registradas: {len(paciente.citas)}")

# Los cambios pasan por interfaces controladas y son validados.
medico.especialidad = "Cardiología"
paciente.estado = "inactivo"
print(f"Nueva especialidad: {medico.especialidad}")
print(f"Nuevo estado: {paciente.estado}")

for operacion in (
    lambda: setattr(medico, "especialidad", ""),
    lambda: setattr(paciente, "estado", "desconocido"),
    lambda: paciente.agendar_cita(medico, datetime(2026, 9, 16, 10, 30)),
):
    try:
        operacion()
    except ValueError as error:
        print(f"Operación rechazada: {error}")

# La cita también cambia de estado únicamente mediante su método público.
paciente.estado = "activo"
paciente.cancelar_cita(cita)
print(f"Estado final de la cita: {cita.estado}")

Especialidad: Medicina interna
Estado inicial: activo
Citas registradas: 1
Nueva especialidad: Cardiología
Nuevo estado: inactivo
Operación rechazada: La especialidad no puede estar vacía
Operación rechazada: El estado debe ser 'activo' o 'inactivo'
Operación rechazada: El paciente inactivo no puede agendar citas
Estado final de la cita: cancelada


## Diagrama UML

```plantuml
@startuml

class Medico {
    - nombre: str
    - __especialidad: str
    - __consultorio: str
    + especialidad: str
    + consultorio: str
}

class Paciente {
    - nombre: str
    - eps: str
    - __estado: str
    - __citas: list[Cita]

    + estado: str
    + citas: tuple[Cita, ...]
    + agendar_cita(medico: Medico, fecha: datetime): Cita
    + cancelar_cita(cita: Cita): None
}

class Cita {
    - fecha: datetime
    - medico: Medico
    - __estado: str

    + estado: str
    + cancelar(): None
}

' Asociación:
' Paciente mantiene las citas en el atributo __citas
Paciente "1" --> "0..*" Cita : administra

' Asociación:
' Cita mantiene permanentemente una referencia al médico
Cita "0..*" --> "1" Medico : asignada a

' Dependencia:
' Paciente recibe Medico como parámetro de agendar_cita()
Paciente ..> Medico : usa para agendar

@enduml
```

### Evidencia del encapsulamiento

- Los atributos `__especialidad`, `__consultorio`, `__estado` y `__citas` no forman parte de la interfaz pública directa.
- `@property` permite consultar el estado de forma segura.
- Los setters de `Medico` y `Paciente` rechazan valores vacíos o estados desconocidos.
- `citas` devuelve una tupla, por lo que el código externo no puede agregar ni eliminar citas desde la colección interna.
- `Cita` no tiene setter para `estado`: solo `cancelar()` puede cambiarlo.